# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL. The FAIR² dataset includes mass spectrometry quantification and annotation features of proteins from human mast cell-derived extracellular vesicles.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

# Install seaborn for visualization
!pip install seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their @id values.

Let's enumerate all record sets and display their IDs, names, and fields. This will help decide which ones to explore.

In [ ]:
# List available record sets and their fields
from pprint import pprint

record_sets = list(metadata.record_sets)
print(f"Found {len(record_sets)} record sets:\n")

for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id} | name: {field.name} | data_type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from the primary protein abundance record set into a DataFrame.

We'll select the main protein quantification table, using its `@id`, and load its data.

In [ ]:
# Choose the main quantitative record set by @id
# (Replace this @id with the one appropriate from the above overview cell once populated)
main_rs_id = None
for rs in metadata.record_sets:
    if 'protein' in rs.name.lower() or 'abundance' in rs.name.lower() or 'main' in rs.name.lower():
        main_rs_id = rs.id
        break
# If not found by known keywords, just take the first
if main_rs_id is None:
    main_rs_id = metadata.record_sets[0].id
print(f"Selected RecordSet @id: {main_rs_id}")

# Load records into DataFrames from all record sets
dataframes = {}
for rs in metadata.record_sets:
    records = list(dataset.records(record_set=rs.id))
    try:
        df = pd.DataFrame(records)
        if len(df) > 0:
            dataframes[rs.id] = df
    except Exception as e:
        print(f"Could not load RecordSet {rs.id} due to: {e}")

print(f"Fields in DataFrame for {main_rs_id}:")
print(dataframes[main_rs_id].columns.tolist())

dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's pick a numeric field (such as total peptide count, coverage, or molecular weight) for basic filtering, normalization, and grouping.

**Note:** Field names are referenced by their `@id` as per the Croissant schema.

In [ ]:
# Display available fields and try to select a common numeric one
df = dataframes[main_rs_id]
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric fields: {numeric_fields}")

# If there are no detected dtypes, try to coerce some likely numeric columns
if not numeric_fields:
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            pass
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f"After conversion, numeric fields: {numeric_fields}")

# Choose the first found numeric field, e.g. coverage, peptide count, or molecular_weight
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    raise ValueError('No numeric fields found in the main RecordSet.')

# Threshold (example: filter for meaningful entries)
threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype != 'O' else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df[[numeric_field_id]].head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field, if available
possible_group_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col]) or len(df[col].unique()) < 16]
group_field = None
for col in possible_group_fields:
    if col.lower() in ['accession', 'description', 'sample', 'group']:
        group_field = col
        break
if not group_field and possible_group_fields:
    group_field = possible_group_fields[0]

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"Grouped average {numeric_field_id} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and relationships between key variables.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id], bins=30, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# If we have a group_field, show top groups by average value
if group_field:
    top_groups = grouped_df.sort_values(numeric_field_id, ascending=False).head(10)
    plt.figure(figsize=(10,4))
    sns.barplot(data=top_groups, x=group_field, y=numeric_field_id)
    plt.title(f'Top groups by average {numeric_field_id}')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Load dataset metadata and records using the mlcroissant library and the dataset's Croissant schema URL.
- Inspect and select record sets and fields using their `@id` values.
- Extract canonical protein quantification data as a DataFrame for analysis.
- Filter, normalize, and group data by key attributes, referencing all entities by their Croissant `@id`.
- Visualize distributions and group summary statistics for protein abundance-related metrics.

This workflow facilitates reproducible, standards-based FAIR data analysis for biomedical mass spectrometry studies.